In [ ]:
import nltk
from gensim.models import Word2Vec
from nltk.corpus import stopwords
import re

In [ ]:
import pandas as pd
data=pd.read_excel(r'C:\Users\harsh\OneDrive\Desktop\Prudhvi new work\full unspsc list.xlsx')
#,names=["Level 4 Description"]

messages

In [ ]:
data.head


In [ ]:
type(data)

In [ ]:
def preprocess_text(text):
    # lowercasing
    lowercased_text = text.lower()

    # cleaning 
    import re 
    remove_punctuation = re.sub(r'[^\w\s]', '', lowercased_text)
    remove_white_space = remove_punctuation.strip()
    text=re.sub(r'\[[0-9]*\]',' ',remove_white_space)
    
    
    # Tokenization = Breaking down each sentence into an array
    from nltk.tokenize import word_tokenize
    tokenized_text = word_tokenize(text)

    # Stop Words/filtering = Removing irrelevant words
    from nltk.corpus import stopwords
    stopwords = set(stopwords.words('english'))
    stopwords_removed = [word for word in tokenized_text if word not in stopwords]

    # Stemming = Transforming words into their base form
    from nltk.stem import PorterStemmer
    ps = PorterStemmer()
    stemmed_text = [ps.stem(word) for word in stopwords_removed]
    
    # Putting all the results into a dataframe.
    df = pd.DataFrame({
        'Level 4 Description': [text],
        'LOWERCASE' : [lowercased_text],
        'CLEANING': [remove_white_space],
        'TOKENIZATION': [tokenized_text],
        'STOP-WORDS': [stopwords_removed],
        'STEMMING': [stemmed_text]
    })

    return df

def preprocessing(corpus):
    # Create an empty DataFrame
    df = pd.DataFrame(columns=['Level 4 Description'])

    # Running preprocessing one by one
    for doc in corpus['Level 4 Description']:
        # Call the preprocess_text function
        result_df = preprocess_text(doc)
        
        # Concatenate the result of preprocessing to the main DataFrame
        df = pd.concat([df, result_df], ignore_index=True)
        
    return df

result_preprocessing = preprocessing(data)
result_preprocessing


In [ ]:
def calculate_tfidf(corpus):
    # Call the preprocessing result
    df = preprocessing(corpus)
        
    # Make each array row from stopwords_removed to be a sentence
    stemming = corpus['STEMMING'].apply(' '.join)
    #stemming = corpus.apply(' '.join)
    
    # Count TF-IDF
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(stemming)
    
    # Get words from stopwords array to use as headers
    feature_names = vectorizer.get_feature_names_out()

    # Combine header titles and weights
    df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
    df_tfidf = pd.concat([df, df_tfidf], axis=1)
    
    return df_tfidf

result_tfidf = calculate_tfidf(result_preprocessing)
result_tfidf

In [ ]:
result_tfidf.describe()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
 def cosineSimilarity(corpus):
    # Call the TF-IDF result
    df_tfidf = calculate_tfidf(corpus)
    
    # Get the TF-IDF vector for the first item (index 0)
    vector1 = df_tfidf.iloc[0, 6:].values.reshape(1, -1)

    # Get the TF-IDF vector for all items except the first item
    vectors = df_tfidf.iloc[:, 6:].values
    
    # Calculate cosine similarity between the first item and all other items
    from sklearn.metrics.pairwise import cosine_similarity
    cosim = cosine_similarity(vector1, vectors)
    cosim = pd.DataFrame(cosim)
    
    # Convert the DataFrame into a one-dimensional array
    cosim = cosim.values.flatten()

    # Convert the cosine similarity result into a DataFrame
    df_cosim = pd.DataFrame(cosim, columns=['COSIM'])

    # Combine the TF-IDF array with the cosine similarity result
    df_cosim = pd.concat([df_tfidf, df_cosim], axis=1)
    
    # Filter the DataFrame to display only rows where COSIM > 0.5
    filtered_df = df_cosim[df_cosim['COSIM'] > 0.5][['Level 4 Description', 'COSIM']]
    print(filtered_df)

    return df_cosim

cosim_result = cosineSimilarity(result_tfidf)
#cosim_result

In [ ]:
cosim_result.describe()

In [ ]:
result_preprocessing['STEMMING']

In [ ]:
result_tfidf.describe()

In [ ]:
############ TRIAL #########################


# Find pairs with cosine similarity > 0.5
similar_pairs = []
for i in range(cosim_result.shape[0]):
    for j in range(i + 1, cosim_result.shape[1]):  # Avoid duplicates and self-similarity
        if cosim_result.iloc[i, j] > 0.5:
            similar_pairs.append((i, j, cosim_result[i, j]))

# Display the results
for i, j, similarity in similar_pairs:
    print(f"Document {i} and Document {j} have cosine similarity of {similarity:.2f}")

In [ ]:
cosim_result.iloc[0,1]

In [ ]:
############## trial code######################
def cosineSimilarity(corpus):
    
    # Step 2: Compute cosine similarity and convert to DataFrame
    cosine_sim_matrix = cosine_similarity(corpus)
    cosim_df = pd.DataFrame(cosine_sim_matrix)

    # Convert all entries in cosim_df to numeric, coercing errors to NaN
    cosim_df = cosim_df.apply(pd.to_numeric, errors='coerce')

    # Step 3: Find pairs with cosine similarity > 0.5
    similar_pairs = []
    for i in range(cosim_df.shape[0]):
        for j in range(i + 1, cosim_df.shape[1]):  # Avoid duplicates and self-similarity
            if pd.notna(cosim_df.iloc[i, j]) and cosim_df.iloc[i, j] > 0.5:  # Check for non-NaN
                similar_pairs.append((i, j, cosim_df.iloc[i, j]))

    # Display the results
    for i, j, similarity in similar_pairs:
        print(f"Document {i} and Document {j} have cosine similarity of {similarity:.2f}")